# Run PBPMCD

This notebook checks the predictor, prepares an included event log, runs PBPMCD, and displays the resulting alarms. Run the cells in order from the repository root.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'pbpmcd').is_dir() else cwd.parent
if not (PROJECT_ROOT / 'pbpmcd').is_dir():
    raise FileNotFoundError('Open this notebook from the PBPMCD repository.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('Project:', PROJECT_ROOT)
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Verify the predictor

The smoke test performs one forward and backward pass on the available CPU or GPU.

In [ ]:
subprocess.run([sys.executable, '-m', 'pbpmcd.smoke_test'], check=True)

## 2. Prepare an included event log

The preparation step groups events by case, orders traces chronologically, creates next-activity prefix-label instances, and writes non-overlapping trace windows.

In [ ]:
dataset = 'cm_gradual_5k'
window_size = 100
epochs = 1  # Use 20 for the standard training configuration.
output_dir = Path('pbpmcd/outputs/notebook_cm_gradual_5k')
source_log = Path('data/gradual_released/cm/cm_gradual_5k.csv')

subprocess.run([
    sys.executable, '-m', 'pbpmcd.prepare_data',
    '--input', str(source_log),
    '--window-sizes', str(window_size),
], check=True)

## 3. Run PBPMCD

The reference positions are the starts of the two gradual intervals recorded in the dataset manifest.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pbpmcd.run_experiment',
    '--dataset', dataset,
    '--window-size', str(window_size),
    '--epochs', str(epochs),
    '--ground-truth', '1000;3000',
    '--output-dir', str(output_dir),
], check=True)

## 4. Inspect the output

In [ ]:
with (output_dir / 'result.json').open(encoding='utf-8') as handle:
    result = json.load(handle)

summary = {
    'raw_alarm_window_indices': result['raw_alarm_window_indices'],
    'alarm_clusters_window_indices': result['alarm_clusters_window_indices'],
    'localized_detections': result['localized_detections'],
    'reported_detections': result['reported_detections'],
    'radius_free': result['primary_protocol']['radius_free'],
}
summary